In [1]:
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

import os
from git import Repo

In [2]:
repo_path = "./repo/review"
repo = Repo.clone_from("https://github.com/fiap-soat10/fiapx-authentication", to_path=repo_path)

In [17]:
loader = GenericLoader.from_filesystem(
    repo_path + "/src/",
    glob="**/*", # Pega todos os arquivos inclusive subpastas
    suffixes = [".cs"],
    exclude = ["**/bin/*", "**/obj/*"],
    parser = LanguageParser(language=Language.CSHARP, parser_threshold=500)
)

documents = loader.load()
len(documents)

36

In [18]:
repo_splitter = RecursiveCharacterTextSplitter.from_language(
    language = Language.CSHARP,
    chunk_size = 2000, # menor que o padrão de 4000 por se tratar de código
    chunk_overlap = 200
)

chunk_code = repo_splitter.split_documents(documents)
len(chunk_code)

66

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [20]:
db = Chroma.from_documents(chunk_code, OpenAIEmbeddings(disallowed_special=()))

retriever = db.as_retriever(
    search_type= "mmr", # Teste de similaridade
    search_kwargs = {"k": 8} # define recuperar 8 chunks relavantes
)

In [21]:
llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=1000)

In [28]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", 
            "Você é um revisor de código experiente. Forneça informações detalhadas sobre a revisão de código e sugestões de melhorias baseada no contexto fornecido abaixo: \n\n{context}"
        ),
        ( "user", "{input}" )
    ]
)

document_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [29]:
response = retrieval_chain.invoke({"input": "Você pode revisar e sugerir melhorias para o código de autenticação"})
print(response['answer'])

**Revisão e Sugestões de Melhorias:**

1. **Login Endpoint:**
   - No método `Login`, considere adicionar validações extras, como verificar se o e-mail e a senha são válidos antes de fazer a autenticação.
   - Poderia ser interessante validar se o e-mail fornecido no request é um e-mail válido antes de prosseguir com o login.

2. **Token Validator Service:**
   - No método `ValidateTokenAndGetUserInfoAsync`, seria útil adicionar tratamento para lidar com exceções específicas relacionadas à validação do token, e fornecer mensagens de erro adequadas para cada caso.

3. **Resend Confirmation Code Endpoint:**
   - No método `ResendConfirmationCode`, sugiro adicionar uma verificação de segurança adicional, pois atualmente parece que qualquer e-mail válido poderia disparar o reenvio do código de confirmação.
   - Considere usar um mecanismo de throttling para limitar o número de solicitações de reenvio de código por um determinado período de tempo.

4. **Token Validation Endpoint:**
   - Ao 